In [1]:
import csv
import json
import re
from difflib import SequenceMatcher
from metrics.hit_checker import parse_ground_truth_movies, calculate_hit_rate_from_json_string


def count_ground_truth_mentions(csv_file_path, similarity_threshold=0.7):
    """
    Count how many entries have the ground truth movie mentioned in the generated conversation.
    
    Args:
        csv_file_path (str): Path to the CSV file
        similarity_threshold (float): Similarity threshold for fuzzy matching (0-1)
        
    Returns:
        dict: Dictionary with counts and details
    """
    total_entries = 0
    successful_mentions = 0
    failed_entries = []
    match_types = {}
    
    try:
        with open(csv_file_path, 'r', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            
            for row in reader:
                total_entries += 1
                ground_truth = row['ground_truth'].strip()
                generated_conversation = row['generated_conversation']
                
                # Parse the generated conversation JSON
                try:
                    
                    # Check if ground truth movie is mentioned
                    hit_score, match_type = calculate_hit_rate_from_json_string(ground_truth, generated_conversation)
                    if hit_score > 0:
                        successful_mentions += 1
                        match_types[match_type] = match_types.get(match_type, 0) + 1
                    else:
                        # Show which movies were being looked for in failed cases
                        movies_searched = parse_ground_truth_movies(ground_truth)
                        failed_entries.append({
                            'dialog_id': row['dialog_id'],
                            'ground_truth': ground_truth,
                            'movies_searched': movies_searched,
                            'reason': match_type
                        })
                        
                except Exception as e:
                    print(f"Error processing dialog_id: {row.get('dialog_id', 'unknown')} - {str(e)}")
                    failed_entries.append({
                        'dialog_id': row.get('dialog_id', 'unknown'),
                        'ground_truth': ground_truth,
                        'error': str(e)
                    })
    
    except FileNotFoundError:
        print(f"Error: File '{csv_file_path}' not found.")
        return None
    except Exception as e:
        print(f"Error reading file: {str(e)}")
        return None
    
    # Calculate percentage
    success_rate = (successful_mentions / total_entries * 100) if total_entries > 0 else 0
    
    results = {
        'total_entries': total_entries,
        'successful_mentions': successful_mentions,
        'failed_mentions': total_entries - successful_mentions,
        'success_rate': success_rate,
        'failed_entries': failed_entries,
        'match_types': match_types
    }
    
    return results

def print_results(results):
    """Print the results in a formatted way."""
    if results is None:
        return
    
    print("=" * 60)
    print("GROUND TRUTH MENTION ANALYSIS (Enhanced)")
    print("=" * 60)
    print(f"Total entries: {results['total_entries']}")
    print(f"Successful mentions: {results['successful_mentions']}")
    print(f"Failed mentions: {results['failed_mentions']}")
    print(f"Success rate: {results['success_rate']:.2f}%")
    print()
    
    if results['match_types']:
        print("Match types breakdown:")
        for match_type, count in results['match_types'].items():
            percentage = (count / results['successful_mentions'] * 100)
            print(f"  - {match_type}: {count} ({percentage:.1f}%)")
        print()
    
    if results['failed_entries']:
        print("Failed entries (first 10):")
        for entry in results['failed_entries'][:10]:
            movies_info = f" (searched: {', '.join(entry['movies_searched'])})" if 'movies_searched' in entry else ""
            error_msg = f" (Error: {entry['error']})" if 'error' in entry else f" (Reason: {entry['reason']})"
            print(f"  - {entry['dialog_id']}: {entry['ground_truth']}{movies_info}{error_msg}")
        
        if len(results['failed_entries']) > 10:
            print(f"  ... and {len(results['failed_entries']) - 10} more")

# Example usage
if __name__ == "__main__":
    # Replace with your actual CSV file path
    csv_file_path = "../generated_testsets/multiturn_test/sft/llama-3.2-instruct/redial/generated_movie_conversations.csv"
    
    # You can adjust the similarity threshold (0.7 = 70% similarity)
    results = count_ground_truth_mentions(csv_file_path, similarity_threshold=0.9)
    print_results(results)

GROUND TRUTH MENTION ANALYSIS (Enhanced)
Total entries: 1076
Successful mentions: 258
Failed mentions: 818
Success rate: 23.98%

Match types breakdown:
  - exact_match (Lethal Weapon): 1 (0.4%)
  - exact_match (It): 23 (8.9%)
  - exact_match (Avengers: Infinity War): 5 (1.9%)
  - exact_match (Spider-Man: Homecoming): 1 (0.4%)
  - exact_match (Billy Madison): 2 (0.8%)
  - exact_match (A Quiet Place): 5 (1.9%)
  - exact_match (Black Sheep): 1 (0.4%)
  - exact_match (The Wedding Singer): 1 (0.4%)
  - exact_match (Captain America: Civil War): 1 (0.4%)
  - exact_match (American Gangster): 2 (0.8%)
  - exact_match (Blazing Saddles): 1 (0.4%)
  - exact_match (Tommy Boy): 1 (0.4%)
  - exact_match (The Others): 1 (0.4%)
  - exact_match (Batman): 1 (0.4%)
  - exact_match (Anchorman): 1 (0.4%)
  - exact_match (Room): 1 (0.4%)
  - exact_match (The Fault in Our Stars): 3 (1.2%)
  - exact_match (The House): 1 (0.4%)
  - exact_match (Deadpool 2): 10 (3.9%)
  - exact_match (Die Hard): 4 (1.6%)
  - exa